# Dataset diverso de ondas gravitacionales + estadística básica

Este notebook construye **un único dataset** desde catálogos públicos de GWOSC (GWTC) y aplica análisis estadístico básico sobre variables físicas de eventos de ondas gravitacionales.

## Objetivos
- Obtener un dataset limpio y útil para análisis.
- Incluir diferentes tipos de eventos (BBH, BNS, NSBH, ambiguos).
- Aplicar estadística descriptiva, correlaciones, visualizaciones e inferencia básica.


In [ ]:
import subprocess
import sys

for pkg in ["gwosc", "pandas", "numpy", "matplotlib", "seaborn", "scipy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pkg], check=False)

print("Dependencias verificadas.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from gwosc.datasets import find_datasets
from gwosc.api.v2 import fetch_event_version, fetch_json

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)


## 1) Construcción del dataset desde GWOSC

Estrategia:
1. Listar catálogos disponibles en GWOSC.
2. Seleccionar catálogos GWTC (priorizando los marcados como `confident`).
3. Extraer parámetros físicos por evento y unificarlos en un `DataFrame`.


In [ ]:
def _to_float(x):
    if x is None:
        return np.nan
    try:
        return float(x)
    except Exception:
        return np.nan


def _classify_event(m1, m2):
    if np.isnan(m1) or np.isnan(m2):
        return "unknown"
    if m1 < 3 and m2 < 3:
        return "BNS"
    if (m1 < 3 <= m2) or (m2 < 3 <= m1):
        return "NSBH"
    if m1 > 5 and m2 > 5:
        return "BBH"
    return "Ambiguous"


def _extract_best_parameters(analysis):
    params = {}
    for p in analysis.get("parameters", []):
        name = p.get("name")
        if not name:
            continue
        value = p.get("best")
        if value is None:
            value = p.get("median")
        params[name] = value
    return params


def build_gwtc_dataset(max_catalogs=4):
    expected_cols = [
        "event", "catalog", "m1", "m2", "chirp_mass", "chi_eff",
        "luminosity_distance_mpc", "network_snr", "redshift",
        "mass_ratio_q", "total_mass", "symmetric_mass_ratio", "event_type"
    ]

    try:
        catalogs = sorted(find_datasets(type="catalog"))
    except Exception as exc:
        print(f"No se pudo conectar a GWOSC para listar catálogos: {exc}")
        return pd.DataFrame(columns=expected_cols)

    gwtc = [c for c in catalogs if c.upper().startswith("GWTC")]
    confident = [c for c in gwtc if "confident" in c.lower()]
    selected = confident[:max_catalogs] if confident else gwtc[:max_catalogs]

    rows = []
    for catalog in selected:
        try:
            events = find_datasets(type="event", catalog=catalog)
        except Exception:
            continue

        for name in events:
            try:
                meta = fetch_event_version(event=name, catalog=catalog)
                params_url = meta.get("parameters_url")
                if not params_url:
                    continue
                if params_url.startswith("/"):
                    params_url = f"https://gwosc.org{params_url}"

                payload = fetch_json(params_url)
                results = payload.get("results", [])
                if not results:
                    continue

                preferred = [r for r in results if r.get("is_preferred")]
                pe = [r for r in results if r.get("pipeline_type") == "pe"]
                analysis = preferred[0] if preferred else (pe[0] if pe else results[0])

                p = _extract_best_parameters(analysis)

                m1 = _to_float(p.get("mass_1_source") or p.get("mass_1"))
                m2 = _to_float(p.get("mass_2_source") or p.get("mass_2"))
                if np.isnan(m1) or np.isnan(m2):
                    continue

                mc = _to_float(p.get("chirp_mass_source") or p.get("chirp_mass") or p.get("mchirp"))
                chi_eff = _to_float(p.get("chi_eff"))
                dL = _to_float(p.get("luminosity_distance") or p.get("distance"))
                snr = _to_float(p.get("network_matched_filter_snr") or p.get("network_snr") or p.get("snr"))
                z = _to_float(p.get("redshift"))

                rows.append({
                    "event": name,
                    "catalog": catalog,
                    "m1": m1,
                    "m2": m2,
                    "chirp_mass": mc,
                    "chi_eff": chi_eff,
                    "luminosity_distance_mpc": dL,
                    "network_snr": snr,
                    "redshift": z,
                })
            except Exception:
                continue

    df = pd.DataFrame(rows).drop_duplicates(subset=["event", "catalog"]).reset_index(drop=True)
    if df.empty:
        print("No se pudo construir el dataset (sin eventos parseados).")
        return pd.DataFrame(columns=expected_cols)

    df["mass_ratio_q"] = df["m2"] / df["m1"]
    df["total_mass"] = df["m1"] + df["m2"]
    df["symmetric_mass_ratio"] = (df["m1"] * df["m2"]) / (df["total_mass"] ** 2)

    z_approx = df["luminosity_distance_mpc"] / 4425.0
    df["redshift"] = df["redshift"].fillna(z_approx)

    df["event_type"] = [_classify_event(a, b) for a, b in zip(df["m1"], df["m2"])]
    return df


In [ ]:
df = build_gwtc_dataset()

print(f"Eventos totales: {len(df)}")
if not df.empty:
    print("
Columnas:", list(df.columns))
    print("
Tipos de evento:")
    print(df["event_type"].value_counts(dropna=False))
    print("
Catálogos usados:")
    print(df["catalog"].value_counts())
    display(df.head(10))
else:
    print("No se pudo construir el dataset. Revisa conectividad o dependencias.")


## 2) Estadística descriptiva y control de calidad

In [ ]:
if not df.empty:
    numeric_cols = [
        "m1", "m2", "chirp_mass", "mass_ratio_q", "total_mass",
        "chi_eff", "luminosity_distance_mpc", "redshift", "network_snr"
    ]

    summary = df[numeric_cols].describe().T
    summary["missing"] = df[numeric_cols].isna().sum()
    summary["missing_%"] = 100 * summary["missing"] / len(df)
    display(summary)
else:
    print("Sin datos: omitiendo estadística descriptiva.")


## 3) Visualización básica

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    sns.histplot(df["chirp_mass"].dropna(), bins=20, kde=True, ax=axes[0], color="C0")
    axes[0].set_title("Distribución de masa chirp")

    sns.histplot(df["network_snr"].dropna(), bins=20, kde=True, ax=axes[1], color="C1")
    axes[1].set_title("Distribución de SNR")

    sns.histplot(df["redshift"].dropna(), bins=20, kde=True, ax=axes[2], color="C2")
    axes[2].set_title("Distribución de redshift")

    plt.tight_layout()
    plt.show()
else:
    print("Sin datos: omitiendo histogramas.")


In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.scatterplot(
        data=df,
        x="m2", y="m1",
        hue="event_type",
        size="network_snr",
        sizes=(20, 180),
        alpha=0.75,
        ax=axes[0],
    )
    axes[0].plot([0, 120], [0, 120], "k--", alpha=0.4)
    axes[0].set_title("Masa secundaria vs primaria")

    sns.boxplot(data=df, x="event_type", y="chirp_mass", ax=axes[1])
    axes[1].set_title("Masa chirp por tipo de evento")
    axes[1].tick_params(axis="x", rotation=20)

    plt.tight_layout()
    plt.show()
else:
    print("Sin datos: omitiendo scatter/boxplot.")


## 4) Correlaciones e inferencia básica

In [ ]:
if not df.empty:
    corr_cols = [
        "m1", "m2", "chirp_mass", "mass_ratio_q", "total_mass",
        "chi_eff", "luminosity_distance_mpc", "redshift", "network_snr"
    ]
    corr = df[corr_cols].corr(numeric_only=True)

    plt.figure(figsize=(10, 7))
    sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
    plt.title("Matriz de correlación (Pearson)")
    plt.show()
else:
    print("Sin datos: omitiendo matriz de correlación.")


In [ ]:
if not df.empty:
    groups = []
    labels = []
    for g, gdf in df.groupby("event_type"):
        vals = gdf["chirp_mass"].dropna().values
        if len(vals) >= 3:
            groups.append(vals)
            labels.append(g)

    if len(groups) >= 2:
        stat, pvalue = stats.kruskal(*groups)
        print("Prueba de Kruskal-Wallis sobre chirp_mass por tipo de evento")
        print("Grupos usados:", labels)
        print(f"Estadístico H = {stat:.3f}")
        print(f"p-valor      = {pvalue:.4g}")
    else:
        print("No hay grupos suficientes para prueba inferencial.")
else:
    print("Sin datos: omitiendo prueba de hipótesis.")


In [ ]:
if not df.empty:
    rng = np.random.default_rng(42)
    x = df["chirp_mass"].dropna().values

    if len(x) > 10:
        B = 2000
        means = np.empty(B)
        n = len(x)
        for i in range(B):
            sample = rng.choice(x, size=n, replace=True)
            means[i] = sample.mean()

        ci_low, ci_high = np.percentile(means, [2.5, 97.5])
        print(f"Media chirp_mass = {x.mean():.3f}")
        print(f"IC bootstrap 95% = [{ci_low:.3f}, {ci_high:.3f}]")

        plt.figure(figsize=(8, 4))
        sns.histplot(means, bins=35, kde=True, color="C4")
        plt.axvline(ci_low, color="k", ls="--", label="IC 2.5%")
        plt.axvline(ci_high, color="k", ls="--", label="IC 97.5%")
        plt.title("Bootstrap de la media de chirp_mass")
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("Pocos datos para bootstrap robusto.")
else:
    print("Sin datos: omitiendo bootstrap.")


## 5) Exportación opcional del dataset

Si deseas reutilizar el dataset fuera del notebook, ejecuta la siguiente celda.


In [ ]:
if not df.empty:
    out_csv = "gwtc_dataset_estadistica_basica.csv"
    df.to_csv(out_csv, index=False)
    print(f"Dataset exportado a: {out_csv}")
else:
    print("Sin datos: no se exportó archivo.")


## Conclusión

Con este dataset puedes practicar estadística básica en un contexto real de ondas gravitacionales:
- estadística descriptiva,
- análisis de distribuciones,
- correlaciones,
- comparación entre grupos,
- e intervalos de confianza por remuestreo.

Extensión sugerida: repetir el análisis por catálogo/observing run para comparar evolución temporal de la población detectada.
